# Chapter 03 — SOLUTIONS: Introduction to Supervised Learning

**Instructor file — share only after exercise session.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(42)

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

## Task 1 Solution: Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])
print('Class proportions in train:', np.bincount(y_train) / len(y_train))
print('Class proportions in test: ', np.bincount(y_test)  / len(y_test))
print('→ stratify=y kept the class balance the same in both sets!')

## Task 2 Solution: Train KNN

In [ ]:
# 2a: Create model
model = KNeighborsClassifier(n_neighbors=5)

# 2b: Train
model.fit(X_train, y_train)

# 2c: Predict
y_pred = model.predict(X_test)

# 2d: Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.4f}')
print(f'→ The model correctly classified {acc*100:.1f}% of test flowers')

## Task 3 Solution: Observing Overfitting

In [ ]:
k_values = [1, 3, 5, 7, 10, 20]
train_accs, test_accs = [], []

print(f'{"k":>4}  {"Train Acc":>10}  {"Test Acc":>10}')
print('-' * 30)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, knn.predict(X_train))
    test_acc  = accuracy_score(y_test,  knn.predict(X_test))
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    print(f'{k:>4}  {train_acc:>10.4f}  {test_acc:>10.4f}')

print('\nNote: k=1 has perfect training accuracy (it memorizes every point)')
print('but may perform worse on test data than larger k values.')

## Task 4 Solution: Cross-Validation

In [ ]:
print('5-Fold Cross-Validation:')
print(f'{"k":>4}  {"CV Mean":>10}  {"CV Std":>8}')
print('-' * 28)

cv_means = []
for k in k_values:
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X, y, cv=5)
    cv_means.append(scores.mean())
    print(f'{k:>4}  {scores.mean():>10.4f}  ±{scores.std():.4f}')

best_k = k_values[np.argmax(cv_means)]
print(f'\nBest k by cross-validation: {best_k}')

## Bonus Solution: Train vs Test Accuracy Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_values, train_accs, 'o-', color='#3498db', label='Train Accuracy', linewidth=2)
ax.plot(k_values, test_accs,  's-', color='#e74c3c', label='Test Accuracy',  linewidth=2)
ax.plot(k_values, cv_means,   '^--', color='#9b59b6', label='CV Accuracy', linewidth=2)
ax.set_xlabel('k (number of neighbors)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('KNN: Train vs Test vs CV Accuracy', fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(k_values)
plt.tight_layout()
plt.show()
print('At k=1: perfect training accuracy (overfitting), then stabilizes.')
print('Cross-validation selects k that best generalizes to unseen data.')